# NY Yellow Taxi Trip vs Weather Analysis
This notebook compares the Yellow Taxi Ride against the Weather to see if any relation could be found. _Data only for January 2026 is compared_. 

### Source of data 
_New York Taxi Trips_ data for yellow taxis was downloaded from: 
https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page$0.
The column names description can be found in yellow taxi data dictionary:
https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf$0.


The _Daily Weather Data for New York Wallstreet Station_ was downloaded from:
https://dev.meteostat.net/data/timeseries/daily$0.
The station ID can be found in the full dump:
https://github.com/meteostat/weather-stations$0.
The column names description can be found in meteorological documentation page:
https://dev.meteostat.net/parameters?g=daily&d=1$0.


In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
dbutils.widgets.dropdown("debug_mode", "False", ["True", "False"])  # noqa
DEBUG = dbutils.widgets.get("debug_mode") == "True"  # noqa
print(f"Debug mode: {DEBUG}")

## Imports

In [0]:
from datetime import datetime

import matplotlib.pyplot as plt
import yaml
from pyspark.sql import functions as F

from repository.spark_repo import SparkRepository
from schemas.input_schema import NEW_YORK_WEATHER_SCHEMA
from transformations.data_cleaning import (
    quarantine_date_range,
    quarantine_negative_values,
    quarantine_null_values,
    timestamp_normalization,
)
from utils.constants import ISO_DATE_FORMAT

## Local Constants

In [0]:
# Range constants
# Datetime Range = [_MIN_DATETIME, _MAX_DATETIME)
_MIN_DATETIME = datetime(2026, 1, 1)
_MAX_DATETIME = datetime(2026, 2, 1)  # datetime to be excluded in the range
# Timezone
_NEW_YORK_TIMEZONE = "America/New_York"

## Load Input Data

In [0]:
try:
    with open("pipeline_config.yaml") as f:
        config = yaml.safe_load(f)
except Exception as e:
    print(f"Error loading config file: {e}")
print(config)

### Load Yellow Taxi Ride Data

In [0]:
taxi_raw_data = SparkRepository(spark).read(  # noqa
    path=config.get("ny_yellow_taxi").get("path"), file_format="parquet"
)

if DEBUG:
    print(f"Number of rows in taxi_raw_data: {taxi_raw_data.count()}")
    display(taxi_raw_data.limit(10))

### Load Weather Data for New York Wallstreet

In [0]:
weather_raw_data = SparkRepository(spark).read(  # noqa
    path=config.get("ny_weather").get("path"),
    file_format="csv",
    header=True,
    schema=NEW_YORK_WEATHER_SCHEMA,
)

if DEBUG:
    print(f"Number of rows in weather_raw_data: {weather_raw_data.count()}")
    display(weather_raw_data.limit(10))

## Clean Input Data

### Cleaning taxi data

In [0]:
taxi_wo_null, taxi_q_null = quarantine_null_values(taxi_raw_data, taxi_raw_data.columns)

if DEBUG:
    display(taxi_wo_null.limit(10))
    display(taxi_q_null.limit(10))
    print(f"Number of rows in taxi_wo_null: {taxi_wo_null.count()}")
    print(f"Number of rows in taxi_q: {taxi_q_null.count()}")

In [0]:
taxi_wo_negative, taxi_q_negative = quarantine_negative_values(
    taxi_wo_null,
    [
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "congestion_surcharge",
        "Airport_fee",
        "cbd_congestion_fee",
    ],
)

if DEBUG:
    display(taxi_wo_negative.limit(10))
    display(taxi_q_negative.limit(10))
    print(f"Number of rows in taxi_wo_negative: {taxi_wo_negative.count()}")
    print(f"Number of rows in taxi_q_negative: {taxi_q_negative.count()}")

In [0]:
taxi_fixed_pickup_datetime = timestamp_normalization(
    taxi_wo_negative, "tpep_pickup_datetime", _NEW_YORK_TIMEZONE, "pickup_datetime"
)
taxi_fixed_datetime = timestamp_normalization(
    taxi_fixed_pickup_datetime, "tpep_dropoff_datetime", _NEW_YORK_TIMEZONE, "dropoff_datetime"
)

if DEBUG:
    display(taxi_fixed_datetime.limit(10))

In [0]:
taxi_valid_dates, taxi_date_invalid = quarantine_date_range(
    taxi_fixed_datetime, _MIN_DATETIME, _MAX_DATETIME, "pickup_datetime_local"
)

if DEBUG:
    display(taxi_valid_dates.limit(10))
    display(taxi_date_invalid.limit(10))
    print(f"Number of rows in taxi_valid_dates: {taxi_valid_dates.count()}")
    print(f"Number of rows in taxi_date_invalid: {taxi_date_invalid.count()}")

Removing all the rows where passenger where 0

In [0]:
taxi_w_passengers = taxi_valid_dates.filter(F.col("passenger_count") > 0)

if DEBUG:
    display(taxi_w_passengers.limit(10))
    print(f"Number of rows in taxi_w_passengers: {taxi_w_passengers.count()}")

In [0]:
clean_taxi_data = taxi_w_passengers

### Cleaning weather data

In [0]:
weather_wo_null, weather_q_null = quarantine_null_values(weather_raw_data, weather_raw_data.columns)

if DEBUG:
    display(weather_wo_null.limit(10))
    display(weather_q_null.limit(10))
    print(f"Number of rows in weather_wo_null: {weather_wo_null.count()}")
    print(f"Number of rows in weather_q_null: {weather_q_null.count()}")

In [0]:
weather_wo_negative, weather_q_negative = quarantine_negative_values(
    weather_wo_null, ["day", "month", "year", "rhum", "prcp", "wspd", "pres", "cldc"]
)

if DEBUG:
    display(weather_wo_negative.limit(10))
    display(weather_q_negative.limit(10))
    print(f"Number of rows in weather_wo_negative: {weather_wo_negative.count()}")
    print(f"Number of rows in weather_q_negative: {weather_q_negative.count()}")

In [0]:
weather_by_date = weather_wo_negative.withColumn(
    "date", F.make_date(F.col("year"), F.col("month"), F.col("day"))
).drop("year", "month", "day")

if DEBUG:
    display(weather_by_date.limit(10))

In [0]:
clean_weather_data = weather_by_date

## Analytics

Fetch only the columns from both taxi and weather data for doing analytics

In [0]:
taxi_avg_trips_by_hour = (
    clean_taxi_data.withColumn(
        "pickup_date", F.to_date("pickup_datetime_local", ISO_DATE_FORMAT)
    )
    .withColumn("pickup_hour", F.hour("pickup_datetime_local"))
    .groupBy("pickup_date", "pickup_hour")
    .agg(F.count("*").alias("trip_count"))
    .groupBy("pickup_hour")
    .agg(F.avg("trip_count").alias("average_trip_count"))
    .orderBy("pickup_hour")
)

if DEBUG:
    display(taxi_avg_trips_by_hour.limit(24))

In [0]:
taxi_pd = taxi_avg_trips_by_hour.toPandas()
plt.figure(figsize=(10, 6))
plt.plot(taxi_pd["pickup_hour"], taxi_pd["average_trip_count"], marker='o')
plt.xlabel("Pickup Hour")
plt.ylabel("Average Trip Count")
plt.title("Average Taxi Trips by Hour")
plt.xticks(range(24))
plt.grid(True)
plt.show()

In [0]:
taxi_rides_by_date = clean_taxi_data.withColumn(
    "pickup_date", F.to_date("pickup_datetime_local", ISO_DATE_FORMAT)
).select(["pickup_date", "passenger_count", "total_amount", "trip_distance"])

if DEBUG:
    display(taxi_rides_by_date.limit(10))
    print(f"There are {taxi_rides_by_date.count()} taxi rides in the dataset.")

In [0]:
taxi_rides_by_day = (
    taxi_rides_by_date.groupBy("pickup_date")
    .agg(
        F.sum("passenger_count").alias("total_passengers"),
        F.round(F.sum("total_amount"), 2).alias("total_amount"),
        F.round(F.sum("trip_distance"), 2).alias("total_distance"),
        F.count("*").alias("trip_count"),
    )
    .withColumnRenamed("pickup_date", "date")
)

if DEBUG:
    display(taxi_rides_by_day.limit(10))

In [0]:
taxi_pd = taxi_rides_by_day.select("date", "total_passengers").orderBy("date").toPandas()
plt.figure(figsize=(12, 6))
plt.plot(taxi_pd["date"], taxi_pd["total_passengers"], marker="o")
plt.xlabel("Pickup Date")
plt.ylabel("Number of Passengers")
plt.title("Number of Passengers by Pickup Date")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
weather_measurement_by_day = clean_weather_data.select(
    ["date", "temp", "rhum", "prcp", "wspd", "pres", "cldc"]
)

if DEBUG:
    display(weather_measurement_by_day.limit(10))

In [0]:
# Join weather and taxi data on date
weather_taxi_corr = (
    weather_measurement_by_day.select("date", "temp")
    .join(taxi_rides_by_day.select("date", "trip_count"), on="date", how="inner")
    .select("temp", "trip_count")
    .toPandas()
)

plt.figure(figsize=(8, 6))
plt.scatter(weather_taxi_corr["temp"], weather_taxi_corr["trip_count"], alpha=0.6)
plt.xlabel("Temperature (temp)")
plt.ylabel("Taxi Trip Count")
plt.title("Correlation between Temperature and Taxi Trip Count")
plt.grid(True)
plt.show()

In [0]:
weather_taxi_prcp_corr = (
    weather_measurement_by_day.select("date", "prcp")
    .join(taxi_rides_by_day.select("date", "trip_count"), on="date", how="inner")
    .select("prcp", "trip_count")
    .toPandas()
)

plt.figure(figsize=(8, 6))
plt.scatter(weather_taxi_prcp_corr["prcp"], weather_taxi_prcp_corr["trip_count"], alpha=0.6)
plt.xlabel("Precipitation Percentage (prcp)")
plt.ylabel("Taxi Trip Count")
plt.title("Correlation between Precipitation Percentage and Taxi Trip Count")
plt.grid(True)
plt.show()